In [1]:
import polars as pl
from pathlib import Path

In [2]:
data_path = Path("../../data/1D_Limb_Data.csv")
data_path.exists()

True

In [3]:
df = pl.read_csv(data_path, null_values=["NA", "26 or 25"]).fill_nan(None)
df = df.with_row_index("original_row")
df.columns

['original_row',
 'jjcdr_no',
 'assessment_day',
 'assessment_month',
 'assessment_year',
 'side',
 'transcribed_age',
 'Sex',
 'QuarterOfYear',
 'YearOfBirth',
 'amputation_day',
 'amputation_month',
 'amputation_year',
 'cause.of.amputation',
 'cause_of_amputation_category',
 'Activity.level',
 'activity.level.clean',
 'stump.end',
 'stump.end.clean',
 'stump.musculature',
 'stump.musculature.clean',
 'sensation',
 'sensation.clean',
 'upper.limb.function',
 'condition.of.other.limb',
 'range.of.motion',
 'joint.condition',
 'general.condition.of.stump',
 'muscle.strength',
 'general.health.condition',
 'old.prosthesis.clean',
 'W1',
 'W2',
 'C1',
 'C2',
 'C3',
 'C4',
 'L1',
 'Shoe.size',
 'shoe.sticker',
 'C5',
 'C6',
 'C7',
 'L2',
 'C8',
 'C9',
 'L3',
 'L.foot',
 'W3',
 'jjcdr_no_new',
 'assessmentdate',
 'amputationdate',
 'DOB',
 'earliest_amputationdate',
 'earliest_amputationdate_norm',
 'calculated_age_at_assessment']

In [4]:
cols = ["W1", "W2", "C1", "C2", "C3", "C4", "L1"]
selection = df.select(cols)

**Excerpt taken from dataset email**

A few notes:

- You'll find "JJCDR_no" and "JJCDR_no_new". I added 10,000 to the jjcdr number for left sided amputations so that I could take the mode cause of amputation for each limb and apply it across all rows for each individual.
- Use "earliest_amputation_date" as the amputation date. I took the earliest date per patient, and applied it across all observations.
- DoB is anonymised to the quarter of the year, and I used the middle month of the quarter to calculate it. DoB data was linked up by JJCDR number, and this value is used to calculate "calculated_age_at_assessment". If the calculated age was more than 10 years different from the age written on the form, I changed the age to NA because it means the form was placed in the wrong folder. Similarly, if a JJCDR number has NA for sex, it also means that they are not listed in the DoB database. 
- I think the prosthetists sometimes put the wrong measurements in certain boxes, so I recommend applying some sort of filter to remove outliers. I personally used quite broad filters of `4 < W1` or `W2 < 15`,  `15 < C1`, `C2`, `C3`, or `C4`, and `2 < L1` in my work, but I have the original values in here so you can filter as you wish. 
- I took the causes of amputation and sorted them into broader categories. Both columns are included.
- I also transcribed word for word, and did not edit the writing/grammar.

In [5]:
filtered = selection.filter(
    (pl.col("W1") > 4).fill_null(True),
    (pl.col("W2") < 15).fill_null(True),
    (pl.col("C1") > 15).fill_null(True),
    (pl.col("C2") > 15).fill_null(True),
    (pl.col("C3") > 15).fill_null(True),
    (pl.col("C4") > 15).fill_null(True),
    (pl.col("L1") > 2).fill_null(True),
)

f"Removed {len(selection) - len(filtered)} rows"

'Removed 87 rows'

In [6]:
filtered = filtered.with_columns(
    n_nulls=pl.sum_horizontal(pl.col(cols).is_null())
)

for i in range(7):
    print(f"Columns with {i} missing value{": " if i == 1 else "s:"} {len(filtered.filter(pl.col("n_nulls") == i))}")

Columns with 0 missing values: 390
Columns with 1 missing value:  1126
Columns with 2 missing values: 933
Columns with 3 missing values: 130
Columns with 4 missing values: 17
Columns with 5 missing values: 3
Columns with 6 missing values: 0


In [7]:
filtered = filtered.filter(pl.col("n_nulls") < 3)

f"Removed {len(df) - len(filtered)} rows"

'Removed 245 rows'

In [8]:
for chunk in filtered.filter(pl.col("n_nulls") < 3).iter_slices(n_rows=512):
    x = chunk.select(cols).to_torch()
    print(x.shape)

torch.Size([512, 7])
torch.Size([512, 7])
torch.Size([512, 7])
torch.Size([512, 7])
torch.Size([401, 7])


In [12]:
cols = ["C1", "C2", "C3", "C4", "L1", "W1", "W2"]
df = (
    pl.read_csv(data_path, null_values=["NA", "26 or 25"])
    .fill_nan(None)
    .with_row_index("idx")
    .select(["idx"] + cols)
    .filter(
        (pl.col("W1") > 4).fill_null(True),
        (pl.col("W2") < 15).fill_null(True),
        (pl.col("C1") > 15).fill_null(True),
        (pl.col("C2") > 15).fill_null(True),
        (pl.col("C3") > 15).fill_null(True),
        (pl.col("C4") > 15).fill_null(True),
        (pl.col("L1") > 2).fill_null(True),
    )
    .with_columns(
        n_nulls=pl.sum_horizontal(pl.col(cols).is_null())
    )
)

for chunk in df.filter(pl.col("n_nulls") <= 0).iter_slices(n_rows=512):
    x = chunk.select(cols).to_torch()
    print(x.shape)

df.columns

torch.Size([390, 7])


['idx', 'C1', 'C2', 'C3', 'C4', 'L1', 'W1', 'W2', 'n_nulls']